# Build an Agent

**remark** - this is a tutorial building a simple agent using only LangChain. LangGraph is for building more advance agents and not covered here. 

**opinion** - This really is a great tutorial on the langchain site. It touches all important concepts in a very clear manner.

https://python.langchain.com/docs/tutorials/agents/#installation

In [ ]:
import sys
import os
import openai

# Use current working directory and go one level up
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

# Now you can import your config
from config import api_key, tavily_api_key

#openai.api_key = api_key
os.environ['OPENAI_API_KEY'] = api_key
os.environ['TAVILY_API_KEY'] = tavily_api_key

## End-to-end agent

The code snippet below represents a fully functional agent that uses an LLM to decide which tools to use. It is equipped with a generic search tool. It has conversational memory - meaning that it can be used as a multi-turn chatbot.

In the rest of the guide, we will walk through the individual components and what each part does - but if you want to just grab some code and get started, feel free to use this!

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent

In [ ]:
# create the agent

# This checkpoint saver stores checkpoints in memory using a defaultdict.
memory = MemorySaver()

# initiate a chat model --> new way
model = init_chat_model("openai:gpt-4o")

search = TavilySearch(max_results=10)
tools = [search]
agent_executor = create_react_agent(model, tools, checkpointer=memory)

In [ ]:
agent_executor

In [ ]:
# Use the agent
config = {"configurable": {"thread_id": "abc123"}}

input_message = {
    "role": "user",
    "content": "Hi, I'm Sacha and I live in Voorschoten in the Netherlands.",
}

In [ ]:
messages = {"messages": [input_message]}

for step in agent_executor.stream(messages, config, stream_mode="values"):
    # by using [-1] we only print the latest new message in the state messages
    step['messages'][-1].pretty_print()  

In [ ]:
input_message = {
    "role": "user",
    "content": "Wat can you find on the internet about Sacha van Weeren who is living in the Netherlands has worked for ING and Rabobank.",
    #"content": "What can you find on the internet about Marjet van Weeren who is living Voorschoten in the Netherlands",
}

messages = {"messages": [input_message]}

for step in agent_executor.stream(messages, config, stream_mode="updates"):
    # by using [-1] we only print the latest new message in the state messages
    for section in step.values():
        for msg in section['messages']:
            msg.pretty_print()


In [ ]:
input_message = {
    "role": "user",
    "content": "Use the tavily_search tool to find todays weather where I live?",
}

for step in agent_executor.stream(
    {"messages": [input_message]}, config, stream_mode="values"
):
    step["messages"][-1].pretty_print()

In [ ]:
tools[0].name

#### State memory

An important input to this Agent is that it was provided with `memory`. For this a `config` was created that is given as input when the agents streams output. This means that all historical chats are stored under the `thread_id` in the `config`. This is history is available under `.get_state(config)` method.

> Note - the below code to access memory also works for agents that are build directly using langgraph.

In [ ]:
state = agent_executor.get_state(config)
for msg in state.values['messages']:
    print(msg.pretty_print())

above you see that the llm remembered that the person is from Voorschoten when it was asked `What's the weather where I live!` 

# Now step by step

## Define tools

We first need to create the tools we want to use. Our main tool of choice will be Tavily - a search engine. We can use the dedicated langchain-tavily integration package to easily use Tavily search engine as tool with LangChain.

In [ ]:
from langchain_tavily import TavilySearch, TavilyCrawl

search = TavilySearch(max_result=2)
search_results = search.invoke("What is the weather in Voorschoten?")
print(search_results)

# If we want, we can create other tools
# Once the we have all the tools we want we put them in a list that we will reference
tools =[search]

https://python.langchain.com/docs/how_to/custom_tools/

## Using Language Models

Next, let's learn how to use a language model to call tools. LangChain supports many different language models that you can use interchangably - select the one you want to use below!

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-4.1", model_provider="openai")

In [ ]:
query = "Hi!"
response = model.invoke([{"role": "user", "content": query}])
response.pretty_print()
# or
# response.content
# response.text

We can now see what it is like to enable this model to do tool calling. In order to enable that we use .bind_tools to give the language model knowledge of these tools

In [ ]:
model_with_tools = model.bind_tools(tools)

We can now call the model. Let's first call it with a normal message, and see how it responds. We can look at both the content field as well as the tool_calls field.

In [ ]:
query = "Hi!"
response = model_with_tools.invoke([{"role": "user", "content": query}])

print(f"Message content: {response.text()}\n")
print(f"Tool calls: {response.tool_calls}")

Now, let's try calling it with some input that would expect a tool to be called.



In [ ]:
query = "Search for the weather in Voorschoten"
response = model_with_tools.invoke([{"role": "user", "content": query}])

print(f"Message content: {response.text()}\n")
print(f"Tool calls: {response.tool_calls}")

We can see that there's now no text content, but there is a tool call! It wants us to call the Tavily Search tool.

**This isn't calling that tool yet** - it's just telling us to. In order to actually call it, we'll want to create our agent.

## Create the agent

Now that we have defined the tools and the LLM, we can create the agent. We will be using LangGraph to construct the agent. Currently, we are using a high level interface to construct the agent, but the nice thing about LangGraph is that this high-level interface is backed by a low-level, highly controllable API in case you want to modify the agent logic.

Now, we can initialize the agent with the LLM and the tools.

Note that we are passing in the `model`, not `model_with_tools`. That is because `create_react_agent` will call `.bind_tools` for us under the hood.

In [ ]:
from langgraph.prebuilt import create_react_agent

agent_executor = create_react_agent(model, tools)

## Run the agent

We can now run the agent with a few queries! Note that for now, these are all stateless queries (it won't remember previous interactions). Note that the agent will return the final state at the end of the interaction (which includes any inputs, we will see later on how to get only the outputs).

First up, let's see how it responds when there's no need to call a tool:

In [ ]:
input_message = {"role": "user", "content": "Hi!"}
response = agent_executor.invoke({"messages": [input_message]})

for message in response["messages"]:
    message.pretty_print()

> Note: agents expect the messages to be stored in a `State` that is a kind of dictionary that has a `messages` key and as value the list of messages

In [ ]:
input_message = {"role": "user", "content": "Search for the weather in SF"}
response = agent_executor.invoke({"messages": [input_message]})

for message in response["messages"]:
    message.pretty_print()

## Streaming Messages

We've seen how the agent can be called with .invoke to get a final response. If the agent executes multiple steps, this may take a while. To show intermediate progress, we can stream back messages as they occur.

In [ ]:
for step in agent_executor.stream({"messages": [input_message]}, stream_mode="values"):
    step["messages"][-1].pretty_print()

In [ ]:
for step, metadata in agent_executor.stream(
    {"messages": [input_message]}, config, stream_mode="messages"
):
    if metadata["langgraph_node"] == "agent" and (text := step.text()):
        print(text, end="|")

## Adding in memory

As mentioned earlier, this agent is stateless. This means it does not remember previous interactions. To give it memory we need to pass in a checkpointer. When passing in a checkpointer, we also have to pass in a `thread_id` when invoking the agent (so it knows which thread/conversation to resume from).

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

agent_executor = create_react_agent(model, tools, checkpointer=memory)

config = {"configurable": {"thread_id": "abc123"}}

for step in agent_executor.stream(
    {"messages": [("user", "Hi, I'm Bob!")]}, config, stream_mode="values"
):
    step["messages"][-1].pretty_print()

In [ ]:
for step in agent_executor.stream(
    {"messages": [("user", "What is my name?")]}, config, stream_mode="values"
):
    step["messages"][-1].pretty_print()

In [ ]:
mem = agent_executor.get_state(config)
print(mem)

If you want to start a new conversation, all you have to do is change the thread_id used

In [ ]:
config = {"configurable": {"thread_id": "xyz123"}}

for step in agent_executor.stream(
    {"messages": [("user", "What is my name?")]}, config, stream_mode="values"
):
    step["messages"][-1].pretty_print()